# Forecast-Vintage Series Inspection

Simple notebook for checking whether one series looks clean in the forecast-vintage dataset built by `utils.forecast_vintages`.

What it shows:
- core registry metadata for the selected series
- source-tag mix and source-tag runs
- largest raw jumps with source tags before and after the jump
- full-history raw and transformed plots
- zoomed windows around the biggest raw jumps
- a dataset-level fault check for heavy missingness or suspicious revision jumps

Usage:
1. Run all cells once.
2. Change `SERIES` or the date range in the config cell.
3. Re-run the summary / quality / plot cells.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

ROOT = Path.cwd().resolve()
if not (ROOT / "utils").exists() and (ROOT.parent / "utils").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.forecast_vintages import ForecastVintageMacroStore, month_end

store = ForecastVintageMacroStore()
registry = store.registry.copy()

print({
    "registry_series": len(store.series_names),
    "first_archive_month": str(store.first_archive_month.date()),
    "last_archive_month": str(store.last_archive_month.date()),
})

In [ ]:
SERIES = "CPIAUCSL"
START_DATE = "1989-01-31"
END_DATE = "2025-12-31"
DROP_SERIES = ("HWI", "HWIURATIO")
TOP_JUMPS = 8
ZOOM_JUMPS = 4
WINDOW_MONTHS = 12
MISSING_SHARE_WARN = 0.20
JUMP_LOG_WARN = 0.75
FAULT_TOP_N = 15


# Build one as-of row per forecast date so the rest of the notebook can inspect
# the new dataset in the same time-series shape as the older CSV-based notebook.
def build_row_panel_for_dates(
    forecast_dates: pd.DatetimeIndex,
    extra_drop_series: tuple[str, ...] = (),
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    raw_rows = []
    transformed_rows = []
    source_rows = []
    metadata_rows = []

    for date in forecast_dates:
        forecast_date = month_end(date)
        panel = store.panel_for_forecast_date(
            forecast_date,
            start=forecast_date,
            end=forecast_date,
            extra_drop_series=extra_drop_series,
        )
        raw_rows.append(panel.raw.loc[[forecast_date]])
        transformed_rows.append(panel.transformed.loc[[forecast_date]])
        source_rows.append(panel.source_tags.loc[[forecast_date]])
        metadata_rows.append(panel.metadata)

    raw_frame = pd.concat(raw_rows).sort_index()
    transformed_frame = pd.concat(transformed_rows).sort_index()
    source_frame = pd.concat(source_rows).sort_index()
    metadata_frame = pd.DataFrame(metadata_rows).sort_values("forecast_date").reset_index(drop=True)
    return raw_frame, transformed_frame, source_frame, metadata_frame


forecast_dates = pd.date_range(month_end(START_DATE), month_end(END_DATE), freq=pd.offsets.MonthEnd(1))
raw, tcode, src, panel_meta = build_row_panel_for_dates(forecast_dates, extra_drop_series=DROP_SERIES)

assert SERIES in raw.columns, f"{SERIES} not found in forecast-vintage outputs"

print({
    "raw_shape": raw.shape,
    "tcode_shape": tcode.shape,
    "src_shape": src.shape,
    "series_count": len(raw.columns),
    "date_min": str(raw.index.min().date()),
    "date_max": str(raw.index.max().date()),
})


def target_series_id(series_name: str) -> str:
    row = registry.loc[registry["mnemonic_hs"] == series_name]
    if row.empty:
        return series_name
    target = row["target_series_id"].iloc[0] if "target_series_id" in row.columns else ""
    if pd.isna(target) or str(target).strip() == "":
        return series_name
    return str(target).strip()


def source_runs(tag_series: pd.Series) -> pd.DataFrame:
    s = tag_series.fillna("MISSING").astype(str)
    groups = s.ne(s.shift()).cumsum()
    rows = []
    for _, seg in s.groupby(groups):
        rows.append(
            {
                "source_tag": seg.iloc[0],
                "start": seg.index[0],
                "end": seg.index[-1],
                "months": len(seg),
            }
        )
    out = pd.DataFrame(rows)
    if not out.empty:
        out["start"] = pd.to_datetime(out["start"])
        out["end"] = pd.to_datetime(out["end"])
    return out


def raw_jump_table(series_name: str, top_n: int = 8) -> pd.DataFrame:
    values = pd.to_numeric(raw[series_name], errors="coerce")
    tags = src[series_name].fillna("MISSING").astype(str)
    df = pd.DataFrame({"value": values, "source_tag": tags}).dropna(subset=["value"])
    df["prev_value"] = df["value"].shift(1)
    df["prev_source_tag"] = df["source_tag"].shift(1)
    df["prev_date"] = pd.Series(df.index, index=df.index).shift(1)
    positive = df[(df["value"] > 0) & (df["prev_value"] > 0)].copy()
    positive["log_jump_abs"] = np.log(positive["value"] / positive["prev_value"]).abs()
    positive["pct_change"] = 100.0 * (positive["value"] / positive["prev_value"] - 1.0)
    positive["source_changed"] = positive["source_tag"] != positive["prev_source_tag"]
    out = (
        positive.sort_values("log_jump_abs", ascending=False)
        .head(top_n)
        .reset_index()
        .rename(columns={"decision_date": "date", "index": "date"})
    )
    return out[[
        "date",
        "prev_date",
        "prev_value",
        "value",
        "pct_change",
        "log_jump_abs",
        "prev_source_tag",
        "source_tag",
        "source_changed",
    ]]


def source_palette(tag_series: pd.Series) -> dict[str, tuple[float, float, float]]:
    tags = list(pd.Index(tag_series.fillna("MISSING").astype(str).unique()))
    colors = sns.color_palette("tab10", n_colors=max(len(tags), 3))
    return {tag: colors[i] for i, tag in enumerate(tags)}


def series_quality_summary(series_name: str) -> pd.DataFrame:
    values = pd.to_numeric(raw[series_name], errors="coerce")
    tags = src[series_name].fillna("MISSING").astype(str)
    jumps = raw_jump_table(series_name, top_n=1)
    first_valid = values.first_valid_index()
    last_valid = values.last_valid_index()
    summary = pd.DataFrame(
        {
            "metric": [
                "series",
                "target_series_id",
                "first_valid",
                "last_valid",
                "non_missing_months",
                "missing_months",
                "missing_share",
                "distinct_source_tags",
                "source_tag_switches",
                "largest_raw_jump_date",
                "largest_raw_jump_log_abs",
            ],
            "value": [
                series_name,
                target_series_id(series_name),
                first_valid,
                last_valid,
                int(values.notna().sum()),
                int(values.isna().sum()),
                float(values.isna().mean()),
                int(tags.nunique()),
                int(tags.ne(tags.shift()).sum() - 1),
                jumps["date"].iloc[0] if not jumps.empty else pd.NaT,
                float(jumps["log_jump_abs"].iloc[0]) if not jumps.empty else np.nan,
            ],
        }
    )
    return summary


def dataset_fault_tables(
    missing_share_warn: float = 0.20,
    jump_log_warn: float = 0.75,
    top_n: int = 15,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows = []
    for series_name in raw.columns:
        values = pd.to_numeric(raw[series_name], errors="coerce")
        tags = src[series_name].fillna("MISSING").astype(str)
        jumps = raw_jump_table(series_name, top_n=1)
        rows.append(
            {
                "series": series_name,
                "missing_months": int(values.isna().sum()),
                "missing_share": float(values.isna().mean()),
                "source_tag_switches": int(tags.ne(tags.shift()).sum() - 1),
                "max_jump_date": jumps["date"].iloc[0] if not jumps.empty else pd.NaT,
                "max_pct_change": float(jumps["pct_change"].iloc[0]) if not jumps.empty else np.nan,
                "max_log_jump_abs": float(jumps["log_jump_abs"].iloc[0]) if not jumps.empty else np.nan,
                "jump_source_changed": bool(jumps["source_changed"].iloc[0]) if not jumps.empty else False,
                "prev_source_tag": jumps["prev_source_tag"].iloc[0] if not jumps.empty else None,
                "source_tag": jumps["source_tag"].iloc[0] if not jumps.empty else None,
            }
        )

    series_faults = pd.DataFrame(rows).sort_values(
        ["missing_share", "max_log_jump_abs"],
        ascending=[False, False],
    )
    missing_faults = series_faults.loc[
        series_faults["missing_share"] >= missing_share_warn,
        ["series", "missing_months", "missing_share", "source_tag_switches"],
    ].head(top_n)
    jump_faults = series_faults.loc[
        series_faults["max_log_jump_abs"] >= jump_log_warn,
        [
            "series",
            "max_jump_date",
            "max_pct_change",
            "max_log_jump_abs",
            "jump_source_changed",
            "prev_source_tag",
            "source_tag",
        ],
    ].head(top_n)
    missing_by_date = (
        raw.isna().mean(axis=1)
        .sort_values(ascending=False)
        .rename("missing_share")
        .head(top_n)
        .rename_axis("forecast_date")
        .reset_index()
    )
    return series_faults, missing_faults, jump_faults, missing_by_date


def plot_overview(series_name: str) -> None:
    values = pd.to_numeric(raw[series_name], errors="coerce")
    transformed = pd.to_numeric(tcode[series_name], errors="coerce")
    tags = src[series_name].fillna("MISSING").astype(str)
    palette = source_palette(tags)
    tag_order = list(palette.keys())
    tag_codes = tags.map({tag: i for i, tag in enumerate(tag_order)})
    change_dates = tags.index[tags.ne(tags.shift())][1:]

    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True, height_ratios=[3.0, 2.0, 1.2])

    axes[0].plot(values.index, values.values, color="0.25", linewidth=1.25, alpha=0.9)
    for tag in tag_order:
        mask = tags == tag
        axes[0].scatter(values.index[mask], values[mask], s=12, color=palette[tag], label=tag, alpha=0.9)
    axes[0].set_title(f"{series_name}: raw forecast-vintage series")
    axes[0].set_ylabel("Raw value")
    axes[0].legend(loc="upper left", ncol=2, frameon=True)

    axes[1].plot(transformed.index, transformed.values, color="#1f77b4", linewidth=1.25)
    axes[1].axhline(0.0, color="0.55", linewidth=0.8, linestyle="--")
    axes[1].set_title(f"{series_name}: transformed forecast-vintage series")
    axes[1].set_ylabel("Tcode value")

    axes[2].scatter(tags.index, tag_codes, c=[palette[t] for t in tags], s=18)
    axes[2].set_title("Source tags over time")
    axes[2].set_yticks(range(len(tag_order)))
    axes[2].set_yticklabels(tag_order)
    axes[2].set_xlabel("Forecast date")

    for ax in axes:
        for dt in change_dates:
            ax.axvline(dt, color="0.88", linewidth=0.8, linestyle="--", zorder=0)

    fig.tight_layout()
    plt.show()


def plot_jump_windows(series_name: str, jump_dates: list[pd.Timestamp], window_months: int = 12) -> None:
    if not jump_dates:
        print("No jump dates to plot.")
        return

    values = pd.to_numeric(raw[series_name], errors="coerce")
    tags = src[series_name].fillna("MISSING").astype(str)
    palette = source_palette(tags)
    fig, axes = plt.subplots(len(jump_dates), 1, figsize=(15, 3.5 * len(jump_dates)), sharex=False)
    if len(jump_dates) == 1:
        axes = [axes]

    for ax, dt in zip(axes, jump_dates):
        start = dt - pd.DateOffset(months=window_months)
        end = dt + pd.DateOffset(months=window_months)
        segment = values.loc[start:end].dropna()
        seg_tags = tags.reindex(segment.index).fillna("MISSING").astype(str)
        ax.plot(segment.index, segment.values, color="0.25", linewidth=1.25)
        for tag, color in palette.items():
            mask = seg_tags == tag
            if mask.any():
                ax.scatter(segment.index[mask], segment[mask], s=18, color=color, label=tag)
        ax.axvline(dt, color="crimson", linewidth=1.2, linestyle="--")
        ax.set_title(f"{series_name}: jump window around {dt.date()}")
        ax.set_ylabel("Raw value")
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        axes[0].legend(handles, labels, loc="upper left", ncol=2, frameon=True)
    axes[-1].set_xlabel("Forecast date")
    fig.tight_layout()
    plt.show()

In [ ]:
series = SERIES
target_id = target_series_id(series)

registry_cols = [
    "mnemonic_hs",
    "description_hs",
    "tcode_hs",
    "construction_type",
    "target_series_id",
    "release_rule",
    "revision_class",
    "pub_lag_mode",
    "pub_lag_months",
    "notes",
]

reg_row = registry.loc[registry["mnemonic_hs"] == series, [c for c in registry_cols if c in registry.columns]]

source_count_table = (
    src[series]
    .fillna("MISSING")
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("source_tag")
    .reset_index(name="months")
)
source_count_table["share_pct"] = (100.0 * source_count_table["months"] / source_count_table["months"].sum()).round(2)

run_table = source_runs(src[series])
jump_table = raw_jump_table(series, top_n=TOP_JUMPS)
archive_tag_mix = (
    panel_meta["archive_source_tag"]
    .fillna("MISSING")
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("archive_source_tag")
    .reset_index(name="forecast_dates")
)
panel_overview = pd.DataFrame(
    {
        "metric": [
            "forecast_dates",
            "series_count",
            "date_min",
            "date_max",
            "selected_series",
            "selected_target_series_id",
        ],
        "value": [
            len(raw.index),
            len(raw.columns),
            raw.index.min(),
            raw.index.max(),
            series,
            target_id,
        ],
    }
)

display(panel_overview)
display(series_quality_summary(series))
display(reg_row)
display(source_count_table)
display(run_table.tail(12))
display(jump_table)
display(archive_tag_mix)

In [ ]:
series_faults, missing_faults, jump_faults, missing_by_date = dataset_fault_tables(
    missing_share_warn=MISSING_SHARE_WARN,
    jump_log_warn=JUMP_LOG_WARN,
    top_n=FAULT_TOP_N,
)

dataset_fault_summary = pd.DataFrame(
    {
        "metric": [
            "dataset_missing_share",
            "series_with_any_missing",
            "series_over_missing_threshold",
            "series_over_jump_threshold",
            "worst_date_missing_share",
            "worst_date",
        ],
        "value": [
            float(raw.isna().mean().mean()),
            int((raw.isna().sum() > 0).sum()),
            int((series_faults["missing_share"] >= MISSING_SHARE_WARN).sum()),
            int((series_faults["max_log_jump_abs"] >= JUMP_LOG_WARN).sum()),
            float(missing_by_date["missing_share"].max()) if not missing_by_date.empty else np.nan,
            missing_by_date["forecast_date"].iloc[0] if not missing_by_date.empty else pd.NaT,
        ],
    }
)

display(dataset_fault_summary)
display(missing_faults if not missing_faults.empty else "No series crossed the missing-share warning threshold.")
display(jump_faults if not jump_faults.empty else "No series crossed the jump warning threshold.")
display(missing_by_date)

In [ ]:
plot_overview(SERIES)

In [ ]:
jump_dates = list(raw_jump_table(SERIES, top_n=ZOOM_JUMPS)["date"])
plot_jump_windows(SERIES, jump_dates, window_months=WINDOW_MONTHS)